# ML-09: Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/An3DCu/ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Week 5 ended with a model, a frozen rule, and a tie between them. This notebook checks whether the
numbers behind that tie hold up. It reads two findings from FlyRank's research paper, then asks the same two questions of my own work.

The two questions, asked twice:

1. **Where does the label come from?** What was measured, when, and out of which population.
2. **What does the validation design test?** Which gap between training and deployment the split
   imitates, and which gap it leaves untested.

The sections run in the order the work happened. Rebuild Week 5 exactly (section 0), read the paper (section 1),
re-run the model under a time-aware split (section 2), hunt leakage on the final feature set (section 3), then
rewrite the boldest claim to fit the evidence (section 4).

Nothing about the data is re-chosen this week: same lane constants, same five contract features,
same metric, same seed. Changing the setup would produce a different claim rather than a check of
the existing one. What does change is *when* the model is trained and *which* rows it is judged on,
and that is the one thing Week 5 said it could not test.

## 0. Setup: rebuild Week 5 before auditing it

An audit that cannot reproduce the number it is auditing is not an audit. This section rebuilds the
Week 5 result from the warehouse and checks it against the receipts committed in w03, w04 and w05:
slice size, base rate, missing-label count, held-out rows, the frozen rule's precision@50, the
chosen forest's precision@50 and AUC, and the blind spot numbers. Every one of them is an `assert`.
Had the rebuild drifted, every before/after below would compare this week's mistake against last
week's number.

The one thing this notebook needs that Week 5 did not is more months. Week 5 used a single month
pair, features from March and label from April, and a time question needs a run of them. So section 0b
pulls a five-month panel covering January through May 2026. June is never opened: it is the panel's
final month and the natural outcome window of any past-to-future label in this lane, so it stays
sealed for the capstone.

In [1]:
import importlib.util, subprocess, sys

for pkg in ("duckdb", "huggingface_hub"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json, os, pathlib, warnings
import duckdb, numpy as np, pandas as pd, sklearn

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 190)
SEED = 42          # same seed as w03, w04 and w05, same split, same numbers on a rerun

print(f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__} | duckdb {duckdb.__version__}")


def load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "environment variable"
    for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        env_file = parent / ".env"
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip().strip("\"'"), "local .env (gitignored)"
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN"), "Colab Secret"
    except Exception:
        pass
    raise RuntimeError("No HF_TOKEN found. Set a Colab Secret named HF_TOKEN (read token).")


HF_TOKEN, token_source = load_hf_token()
print(f"HF read token loaded from: {token_source}")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"


def hf_sql(query, tries=6, base_wait=20):
    import time
    for attempt in range(tries):
        try:
            return con.sql(query).df()
        except Exception as exc:
            if "429" not in str(exc) or attempt == tries - 1:
                raise
            wait = base_wait * (attempt + 1)
            print(f"  hub returned HTTP 429, waiting {wait}s, retry {attempt + 1}/{tries - 1}")
            time.sleep(wait)


def fact(month):
    """One month partition of fact_content_daily_performance."""
    return f"read_parquet('{REL}/fact_content_daily_performance/month={month}/*.parquet')"


# --- lane constants: copied unchanged from the w03 contract / w04 baseline / w05 model ---
FEATURE_MONTH = "2026-03"          # Week 5's feature month
LABEL_MONTH   = "2026-04"          # Week 5's label month
DECISION_DATE = "2026-03-31"       # the moment the editor scores the queue
MIN_HISTORY   = "2026-01-01"       # client must have GSC history starting on/before this
MIN_IMP       = 100                # a page needs measurable search demand in the feature month
DECLINE_DROP  = 0.20               # "declining" = next month's impressions >20% below this month's
TOP_K         = 50                 # editorial capacity: the queue is 50 pages long

CONTRACT_FEATURES = ["log_imp_mar", "active_days", "pos_mar", "ctr_mar", "momentum_in_month"]
FEATURES = CONTRACT_FEATURES       # nothing is added this week

# The months this audit may touch. 2026-06 is the panel's final month and stays SEALED.
MONTHS = ["2026-01", "2026-02", "2026-03", "2026-04", "2026-05"]
SEALED_MONTH = "2026-06"
# Mid-month cut for the within-month momentum feature. 2026-03 keeps w03's exact constant (the
# 17th), and shorter months take the day that divides them most evenly. These are NOT the same
# split: a 31-day month divides 16/15, February divides 14/14, April 15/15. Section 2c measures
# what that asymmetry is worth and tests whether it moves any verdict in this notebook.
MID = {"2026-01": "2026-01-17", "2026-02": "2026-02-15", "2026-03": "2026-03-17",
       "2026-04": "2026-04-16", "2026-05": "2026-05-17"}
DAYS_IN_MONTH = {"2026-01": 31, "2026-02": 28, "2026-03": 31, "2026-04": 30, "2026-05": 31}

PAIRS = [("2026-01", "2026-02"), ("2026-02", "2026-03"),
         ("2026-03", "2026-04"), ("2026-04", "2026-05")]   # (feature month, label month)

receipts = {"lane": "bleed_tracker", "seed": SEED, "top_k": TOP_K,
            "sklearn_version": sklearn.__version__, "contract_features": CONTRACT_FEATURES,
            "months_opened": MONTHS, "sealed_month": SEALED_MONTH}


def precision_at_k(df, score_col, label_col="label_declining", k=TOP_K):
    """w04's metric, unchanged: of the top k rows this score ranks highest, how many declined?"""
    return float(df.nlargest(k, score_col)[label_col].mean())


def precision_at_k_fair(df, score_col, label_col="label_declining", k=TOP_K):
    """The same metric, but ties are not allowed to choose the queue (w05's version, unchanged)."""
    s = df[score_col].to_numpy()
    y = df[label_col].to_numpy()
    cut = np.sort(s)[::-1][k - 1]
    above, tied = s > cut, s == cut
    slots = k - int(above.sum())
    fair = (y[above].sum() + slots * y[tied].mean()) / k
    return {"as_listed": float(df.nlargest(k, score_col)[label_col].mean()),
            "tie_fair": float(fair), "tied_at_cut": int(tied.sum()), "slots_from_ties": int(slots)}


def p50(df, score_col):
    """Shorthand: the tie-fair precision@50 this lane ships."""
    return precision_at_k_fair(df, score_col)["tie_fair"]


print(f"\nmonths this notebook may open: {', '.join(MONTHS)}")
print(f"month it must not open: {SEALED_MONTH} (sealed final month, reserved for the capstone)")

python 3.12.6 | numpy 1.26.4 | pandas 2.3.3 | scikit-learn 1.9.0 | duckdb 1.5.4
HF read token loaded from: local .env (gitignored)

months this notebook may open: 2026-01, 2026-02, 2026-03, 2026-04, 2026-05
month it must not open: 2026-06 (sealed final month, reserved for the capstone)


### 0b. Five months of the same slice

One query per month, identical in shape to w03's. Page-level GSC aggregates for every day the
warehouse reports as available, grouped to one row per client × content × month, ordered by
`content_hash_id`. That ordering matters: a parallel `GROUP BY` otherwise returns a different row
order on every rerun, and w05 section 0b measured what that costs, since bootstrap draws and tie-breaks
both move with it.

Two filters that w03 put in SQL now live in pandas. The `>= 100 impressions` threshold and the
client-history filter decide which pages get *scored*, not which pages can be looked up as a label.
Running them inside the monthly query would make a page that fell below 100 impressions next month
come back as missing and count as a total loss. So the panel is pulled unfiltered and the
population rules apply to the feature month only, which is what w05's two separate queries did,
generalised to four month pairs.

In [2]:
panel_parts = []
for m in MONTHS:
    q = f"""
        SELECT '{m}' AS month,
               f.client_hash_id,
               f.content_hash_id,
               SUM(f.gsc_impressions)                                            AS imp,
               SUM(f.gsc_clicks)                                                 AS clk,
               COUNT(*) FILTER (f.gsc_impressions > 0)                           AS active_days,
               SUM(f.gsc_sum_position)                                           AS sum_pos,
               MAX(f.gsc_impressions)                                            AS max_day_imp,
               SUM(f.gsc_impressions) FILTER (f.report_date >= DATE '{MID[m]}')  AS imp_late,
               SUM(f.gsc_impressions) FILTER (f.report_date <  DATE '{MID[m]}')  AS imp_early
        FROM {fact(m)} f
        WHERE f.gsc_data_available IS TRUE
        GROUP BY 1, 2, 3
        ORDER BY f.content_hash_id
    """
    part = hf_sql(q)
    assert not part.content_hash_id.duplicated().any(), f"content_hash_id is not unique in {m}"
    panel_parts.append(part)
    print(f"{m}: {len(part):>7,} page-months pulled")

panel = pd.concat(panel_parts, ignore_index=True)
eligible_clients = set(hf_sql(
    f"SELECT client_hash_id FROM {DIM_CLIENTS} WHERE gsc_data_start <= DATE '{MIN_HISTORY}'"
).client_hash_id)
print(f"\npanel: {len(panel):,} page-months | clients with GSC history from before {MIN_HISTORY}: "
      f"{len(eligible_clients)} of 104")


def build_pair(feature_month, label_month):
    """One decision row per page: features from feature_month, label from label_month.

    The population rules apply to the feature month only. The label lookup is deliberately
    unfiltered, because a page that simply stops appearing next month is a real outcome, not a missing
    value, and section 3c measures how many of those there are and what they do to the numbers.
    """
    f = panel[(panel.month == feature_month)
              & panel.client_hash_id.isin(eligible_clients)
              & (panel.imp >= MIN_IMP)].copy()
    lab = (panel.loc[panel.month == label_month, ["content_hash_id", "imp"]]
                .rename(columns={"imp": "imp_next"}))
    fr = f.merge(lab, on="content_hash_id", how="left")
    assert len(fr) == len(f), "the label join changed the row count"

    fr["seen_next_month"] = fr.imp_next.notna().astype(int)
    fr["imp_next"] = fr.imp_next.fillna(0)

    # --- the five contract features, rebuilt exactly as in w03 ---
    fr["imp_mar"] = fr.imp                                  # kept under w03's column name
    fr["pos_mar"] = fr.sum_pos / fr.imp
    fr["ctr_mar"] = fr.clk / fr.imp
    fr["momentum_in_month"] = fr.imp_late / fr.imp_early.replace(0, np.nan)
    fr["has_first_half_traffic"] = fr.momentum_in_month.notna().astype(int)   # context, not a feature
    fr["momentum_in_month"] = fr.momentum_in_month.fillna(1.0)
    fr["log_imp_mar"] = np.log1p(fr.imp)
    fr["spike_share"] = fr.max_day_imp / fr.imp                               # review column, not a feature

    # --- the label: the only column measured after the decision moment ---
    fr["label_declining"] = (fr.imp_next < (1 - DECLINE_DROP) * fr.imp).astype(int)
    fr["lost_impressions"] = (fr.imp - fr.imp_next).clip(lower=0)              # reporting only
    fr["feature_month"], fr["label_month"] = feature_month, label_month
    return fr.reset_index(drop=True)


pairs = {f"{a}->{b}": build_pair(a, b) for a, b in PAIRS}
overview = pd.DataFrame([{
    "decision_row": k, "pages": len(v), "clients": v.client_hash_id.nunique(),
    "base_rate": v.label_declining.mean(),
    "no_next_month_row": int((1 - v.seen_next_month).sum()),
} for k, v in pairs.items()])
print("\nthe four decision months this lane has (June sealed):")
print(overview.round(4).to_string(index=False))

2026-01: 121,544 page-months pulled


2026-02: 153,559 page-months pulled


2026-03: 176,738 page-months pulled


2026-04: 194,760 page-months pulled


2026-05: 237,910 page-months pulled



panel: 884,511 page-months | clients with GSC history from before 2026-01-01: 40 of 104



the four decision months this lane has (June sealed):
    decision_row  pages  clients  base_rate  no_next_month_row
2026-01->2026-02  63909       28     0.2512                124
2026-02->2026-03  72745       28     0.2290               3610
2026-03->2026-04  85453       27     0.5378                380
2026-04->2026-05  85518       27     0.5377                435


### 0c. The Week 5 result, recomputed and checked against the committed receipts

Same slice, same `GroupShuffleSplit(test_size=0.3, random_state=42)` grouped by client, same frozen
rule, same chosen forest of 300 trees at `min_samples_leaf=400`. Nine asserts, and they are what
makes everything below an audit rather than a rerun.

In [3]:
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ---------------- w04's rule, frozen: nothing below this line was retuned ----------------
POS_BANDS, POS_NAMES = [0, 3, 10, 20, np.inf], ["1-3", "4-10", "11-20", "21+"]
SLIP_GATE, POS_REACH, W_UNDERCLICK, W_REACHABLE = 0.8, 20, 1.0, 0.5


def band_median_ctr(df):
    """The rule's only fitted quantity, learned on training rows only."""
    return (df.assign(pos_band=pd.cut(df.pos_mar, POS_BANDS, labels=POS_NAMES))
              .groupby("pos_band", observed=True)["ctr_mar"].median())


def score_pages(df, band_med_ctr):
    """w04's baseline action score, unchanged."""
    band       = pd.cut(df.pos_mar, POS_BANDS, labels=POS_NAMES)
    severity   = (1 - df.momentum_in_month / SLIP_GATE).clip(lower=0)
    underclick = (df.ctr_mar < band.map(band_med_ctr).astype(float)).astype(int)
    reachable  = ((df.pos_mar >= 1) & (df.pos_mar <= POS_REACH)).astype(int)
    return severity * np.log1p(df.imp_mar) * (1 + W_UNDERCLICK * underclick) * (1 + W_REACHABLE * reachable)
# ----------------------------------------------------------------------------------------

CHOSEN_PARAMS = {"n_estimators": 300, "min_samples_leaf": 400, "n_jobs": -1, "random_state": SEED}
make_forest = lambda: RandomForestClassifier(**CHOSEN_PARAMS)

frame = pairs[f"{FEATURE_MONTH}->{LABEL_MONTH}"].copy()
tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
                      .split(frame, groups=frame.client_hash_id))
frame["split"] = "train"
frame.loc[frame.index[te_idx], "split"] = "test"
TRAIN_CLIENTS = set(frame.loc[frame.split == "train", "client_hash_id"])
TEST_CLIENTS  = set(frame.loc[frame.split == "test",  "client_hash_id"])

train, test = frame[frame.split == "train"].copy(), frame[frame.split == "test"].copy()
BAND_MED_CTR = band_median_ctr(train)
for part in (frame, train, test):
    part["action_score"] = score_pages(part, BAND_MED_CTR)
test["model"] = make_forest().fit(train[FEATURES], train.label_declining).predict_proba(test[FEATURES])[:, 1]

W05_MODEL_P50 = p50(test, "model")
W05_RULE_P50  = p50(test, "action_score")
W05_AUC       = roc_auc_score(test.label_declining, test.model)
blind_w05     = test[test.action_score == 0]

print(f"slice     : {len(frame):,} pages / {frame.client_hash_id.nunique()} clients | "
      f"base rate {frame.label_declining.mean():.2%}")
print(f"held out  : {len(test):,} pages / {len(TEST_CLIENTS)} clients | base rate {test.label_declining.mean():.2%}")
print(f"frozen rule precision@{TOP_K}: {W05_RULE_P50:.1%} | chosen forest: {W05_MODEL_P50:.1%} | "
      f"forest AUC {W05_AUC:.4f}")
print(f"rule's blind spot: {len(blind_w05):,} pages, {int(blind_w05.label_declining.sum()):,} declines, "
      f"base rate {blind_w05.label_declining.mean():.1%}, forest precision@{TOP_K} there "
      f"{p50(blind_w05, 'model'):.1%}")

out_dir = pathlib.Path("../outputs")
w03 = json.loads((out_dir / "w03_data_contract_receipts.json").read_text(encoding="utf-8"))
w04 = json.loads((out_dir / "w04_baseline_score_receipts.json").read_text(encoding="utf-8"))
w05 = json.loads((out_dir / "w05_model_receipts.json").read_text(encoding="utf-8"))

assert len(frame) == w03["frame_rows"], "slice size drifted from the w03 contract"
assert round(float(frame.label_declining.mean()), 4) == w03["label_base_rate"], "base rate drifted"
assert int((1 - frame.seen_next_month).sum()) == w03["pages_missing_april"], "missing-label count drifted"
assert len(test) == w04["test_rows"], "the held-out rows are not last week's held-out rows"
assert round(float(test.label_declining.mean()), 4) == w04["test_base_rate"], "test base rate moved"
assert round(W05_RULE_P50, 4) == w04["baseline_precision_at_50_test"], "the frozen baseline moved"
assert round(W05_MODEL_P50, 4) == w05["comparison_test"]["random forest (chosen)"], "the w05 model score moved"
assert round(W05_AUC, 4) == w05["comparison_auc"]["random forest (chosen)"], "the w05 model AUC moved"
assert len(blind_w05) == w05["blind_spot"]["pages"], "the blind spot moved"
assert round(p50(blind_w05, "model"), 4) == w05["blind_spot"]["chosen_model_precision_at_50"]

print("\nasserts passed: w03 slice size, w03 base rate, w03 missing-label count, w04 held-out rows,")
print("w04 test base rate, w04 frozen precision@50, w05 model precision@50, w05 model AUC,")
print("w05 blind-spot size and blind-spot score. Week 5 reproduces exactly from the warehouse.")
print("Everything below audits *that* result, not a lookalike of it.")

slice     : 85,453 pages / 27 clients | base rate 53.78%
held out  : 25,771 pages / 9 clients | base rate 49.36%
frozen rule precision@50: 90.0% | chosen forest: 90.0% | forest AUC 0.6532
rule's blind spot: 16,556 pages, 6,746 declines, base rate 40.7%, forest precision@50 there 70.0%

asserts passed: w03 slice size, w03 base rate, w03 missing-label count, w04 held-out rows,
w04 test base rate, w04 frozen precision@50, w05 model precision@50, w05 model AUC,
w05 blind-spot size and blind-spot score. Week 5 reproduces exactly from the warehouse.
Everything below audits *that* result, not a lookalike of it.


## 1. Two paper findings + my methodology questions

*Two findings from `docs/flyrank-seo-research-march-2026.pdf`. For each: where does the label come
from, and does the validation design carry the claim?*

The paper sets its own standard on page 4 and holds to it. Headline findings are direct aggregate
comparisons, ML pages are marked exploratory, page 36 states plainly that no p-values or confidence
intervals are reported, and the limitations name the observational design. The point of what
follows is not to catch it out. I picked two findings myself and put to them the same questions
that sections 2 and 3 put to my own model, then described what would let each claim carry further
using the paper's own method.

### Finding #1: "The Anatomy of Growing Content" (page 6, tagged CONFIRMED)

**What it says.** Pages whose impressions are trending up are 37.6% longer, at 3,180 against 2,311
average words, and 20% younger, at 184 against 230 average days, than pages trending down. The
arithmetic checks out, the direction is consistent across two independent columns, and the cohorts
are large.

**Where the label comes from.** The up/down split is `trend_direction`, defined on page 5 as the
30-day versus previous-30-day impression change, with up above 10% growth and down above 10%
decline. The label is therefore a change measured over a 60-day window ending at the snapshot. Word
count and age come from somewhere else. Page 3 says inventory fields such as content count, age and
word count "remain metadata for the full cached portfolio", meaning they are read at export time.

**My question: were the features knowable before the label window opened?** If word count is read
at export, it is measured after the 60 days that decided the label. The paper's own recommended
action for a struggling page is to refresh and expand it, in page 6's "What to Do With This" and
page 34's checklist, and freshness is tracked separately, so pages really do get longer inside
windows like this one. Two readings then fit the same table equally well: longer pages grew, or
growing pages got expanded. The paper is careful to call the comparison observational. My question
is narrower and fixable: as of when is the word count?

**A second question: is the middle moving, or the tail?** Both figures are means, and word count is
long-tailed. The paper already computes percentiles elsewhere: page 30 reports P10 to P90 for eight metrics,
and page 36 names `APPROX_QUANTILES` as the method. A median and a P25/P75 for each cohort would be
that same method applied one row further, and would show whether the typical growing page is
longer or whether a tail of very long pages is carrying the average.

**What would make it carry.** Three additions, all within reach of the existing method:

1. Word count snapshotted at the start of the trend window, or the comparison restricted to pages
   with no update inside it. The freshness field already identifies those.
2. Medians alongside means.
3. One line of scope. The cohorts are 74,187 plus 45,272 pages, which the cell below puts at 35% of
   the 341,701 in the study. The finding describes pages that made a decisive 30-day move, while
   the other 65%, the stable and flat and new, sit outside the comparison.

None of this changes the direction. All three would tell a reader how far to carry it.

### Finding #2: "The Content Performance Curve" (page 7, tagged CONFIRMED)

**What it says.** Health score by content-age bucket climbs to a peak of 33 at 61 to 90 days, falls
to a decay cliff of 14 at 271 to 365 days, then rebounds to 25 at 365 and over. The paper reads
this as a lifecycle of discovery, maturation plateau, decay and recovery, and already narrows the
last step itself: "older pages can recover when they are updated well. This is not evidence that
age naturally reverses performance decline on its own."

**Where the label comes from.** The health score is the hand-built composite defined on page 5:
impressions 30 points, position 30, CTR 20, scroll depth 20, over a rolling 90-day window. The
paper says in the same breath that it is a product score rather than a market outcome.

**My question: is this one cohort followed through its life, or ten groups of pages measured
once?** A lifecycle describes a path a page walks over time. The evidence is a cross-section, with
pages of different ages all measured on the same day. The two coincide only if today's 365-day-old
pages are what today's 75-day-old pages will become. Page 3 describes a portfolio spanning newly
published pages, established evergreen assets and underperforming long-tail pages, which is a mix
of intakes rather than one cohort ageing together, so the old bucket and the young bucket need not
be the same population caught at two ages.

**A second question: which pages could not appear in the 365+ bucket at all?** Page 4's scope rule
keeps the active-content subset to `impressions_90d > 0 and sessions_90d > 0`. A page created 400
days ago that lost all its search demand, or was deleted, is not in that bucket to pull the average
down. The rebound is therefore measured on pages that reached 365 days *and still had traffic*,
which is a different question from what happens to a page as it ages. The paper's own caveat points
this way; the number underneath it remains a survivor average.

**One timing detail.** The health window is 90 days rolling. For a page in the 0 to 7 bucket that
window covers a few days of its life, and for a 365+ page it covers the last quarter. Part of the
early climb is the measurement window filling up.

**What would make it carry.** Follow one creation cohort forward through the monthly series that
page 4 already lists, and publish a survival count per bucket: how many pages entering at 61 to 90
days are still in the active set at 271 to 365, and at 365 and over. Then split the top bucket by
refreshed against not refreshed, which is the reading the paper's own sentence already prefers.
With survival counts beside it, the same chart would support the lifecycle claim instead of
implying it.

In [4]:
# The paper's own published numbers, checked with arithmetic. Nothing here touches private data:
# every figure below is printed in the public PDF (pages 1-7, 30, 36).
PAPER = {
    "portfolio_pieces": 341_701, "brands": 57, "ml_records": 61_790,
    "f1_up_prose": 74_187, "f1_down_prose": 45_272,      # page 6, body text
    "f1_up_table": 74_800, "f1_down_table": 45_600,      # page 6, table row ("74.8K" / "45.6K")
    "f1_words_up": 3_180, "f1_words_down": 2_311,
    "f1_age_up": 184, "f1_age_down": 230,
}

cohort = PAPER["f1_up_prose"] + PAPER["f1_down_prose"]
print("FINDING #1: the arithmetic the paper states")
print(f"  words   : {PAPER['f1_words_up']:,} vs {PAPER['f1_words_down']:,} -> "
      f"+{PAPER['f1_words_up'] / PAPER['f1_words_down'] - 1:.1%} (paper says +37.6%)")
print(f"  age     : {PAPER['f1_age_up']}d vs {PAPER['f1_age_down']}d -> "
      f"-{1 - PAPER['f1_age_up'] / PAPER['f1_age_down']:.1%} younger (paper says 20%)")
print("  both restate correctly. The claim is arithmetically sound; my questions are about"
      " where the label comes from and how much of the portfolio it covers.")

print("\nFINDING #1: how much of the study is inside this comparison?")
print(f"  up {PAPER['f1_up_prose']:,} + down {PAPER['f1_down_prose']:,} = {cohort:,} pieces")
print(f"  study population: {PAPER['portfolio_pieces']:,} pieces across {PAPER['brands']} brands")
print(f"  coverage: {cohort / PAPER['portfolio_pieces']:.1%}, the remaining "
      f"{1 - cohort / PAPER['portfolio_pieces']:.1%} is stable / flat / new and is not compared")

print("\nFINDING #1: one number reported twice, two different ways (page 6)")
for side, prose, table in [("up", PAPER["f1_up_prose"], PAPER["f1_up_table"]),
                           ("down", PAPER["f1_down_prose"], PAPER["f1_down_table"])]:
    print(f"  {side:<5}: table says {table/1000:.1f}K, body text says {prose:,} "
          f"(which rounds to {prose/1000:.1f}K), a gap of {abs(table - prose):,}")
print("  Not material to the direction. One line saying which cut each figure comes from"
      " would stop a checking reader here.")

# Finding #2: the age curve as the paper prints it, with the question a reader cannot answer from it.
age_curve = pd.DataFrame({
    "age_bucket": ["0-7", "8-14", "15-30", "31-60", "61-90", "91-120", "121-180", "181-270",
                   "271-365", "365+"],
    "health_score": [7, 17, 29, 30, 33, 26, 29, 21, 14, 25],
})
age_curve["pages_still_in_the_active_set"] = "not reported"
print("\nFINDING #2: the curve as printed (page 7), plus the column that would settle it")
print(age_curve.to_string(index=False))
peak = age_curve.loc[age_curve.health_score.idxmax()]
cliff = age_curve[age_curve.age_bucket == "271-365"].iloc[0]
last = age_curve.iloc[-1]
print(f"\n  the shape the paper reads as a lifecycle: peak {peak.health_score} at {peak.age_bucket} days"
      f" -> decay cliff {cliff.health_score} at {cliff.age_bucket} -> rebound {last.health_score} at "
      f"{last.age_bucket}")
print("  Each row is a different set of pages measured on the same day, not one cohort followed"
      " through time.")
print("  Without a survival count per bucket, a reader cannot tell a recovery from a survivor"
      " average.")

FINDING #1: the arithmetic the paper states
  words   : 3,180 vs 2,311 -> +37.6% (paper says +37.6%)
  age     : 184d vs 230d -> -20.0% younger (paper says 20%)
  both restate correctly. The claim is arithmetically sound; my questions are about where the label comes from and how much of the portfolio it covers.

FINDING #1: how much of the study is inside this comparison?
  up 74,187 + down 45,272 = 119,459 pieces
  study population: 341,701 pieces across 57 brands
  coverage: 35.0%, the remaining 65.0% is stable / flat / new and is not compared

FINDING #1: one number reported twice, two different ways (page 6)
  up   : table says 74.8K, body text says 74,187 (which rounds to 74.2K), a gap of 613
  down : table says 45.6K, body text says 45,272 (which rounds to 45.3K), a gap of 328
  Not material to the direction. One line saying which cut each figure comes from would stop a checking reader here.

FINDING #2: the curve as printed (page 7), plus the column that would settle it
age_buck

### What these two questions become for my own work

Both questions have a version pointed at my lane, and the rest of the notebook answers them.

| Asked of the paper | Asked of my Week 5 model | Answered in |
|---|---|---|
| Was the feature knowable before the label window opened? | Are all five contract features measured strictly before 2026-03-31, and does anything from April touch them? | sections 3a and 3b |
| Which pages could not appear in the comparison at all? | Which pages and which clients does my slice condition on, and does any of that use the outcome window? | Section 3c |
| Is this one cohort over time, or a cross-section? | My whole result rests on one month pair. Does it hold when the model trains in one month and deploys in the next? | Section 2 |

Week 5 flagged the third row and could not test it. Its section 2 said the grouped split tests a new
client rather than a new month, and that "anything that moved search broadly in April 2026 is
inside every number in this notebook." Section 2 below opens that gap.

## 2. My model under an honest split (before/after)

Week 5's split was already grouped by client, which honestly answers one question: how does this
rank pages for a client the model has never served? It says nothing about the other question, which
is how it ranks pages in a month nobody has lived through yet. The tie, the blind spot finding and
the stability check across 25 re-cuts were all measured inside a single March to April transition,
and 25 re-cuts of one month pair are 25 draws from the same month.

The "after" here is time. Train the identical pipeline on one month pair, then deploy it unchanged
on the next. Four decision months sit below the sealed June, which gives three forward deployments.

One warning about reading the ladder below. Moving from a grouped split to a forward split does not
tighten everything at once; it swaps which constraint is enforced. A forward run that scores every eligible
client has *relaxed* the client constraint, so it is not simply the honest version of Week 5. Only
the last rung tightens both, training on the 18 Week 5 training clients in month t and judging on
the 9 held-out clients in month t+1. That rung is the number an editor should be shown.

In [5]:
# --- what the label is doing across the four decision months, before any model is involved ---
tide = (panel[panel.client_hash_id.isin(eligible_clients)]
        .groupby("month").agg(pages=("content_hash_id", "size"), impressions=("imp", "sum")))
tide["days"] = [DAYS_IN_MONTH[m] for m in tide.index]
tide["imp_per_day"] = (tide.impressions / tide.days).astype(int)
tide["imp_per_day_vs_prev"] = tide.imp_per_day / tide.imp_per_day.shift()
print("the panel's own tide: every eligible client, every page, no model in sight")
print(tide[["pages", "imp_per_day", "imp_per_day_vs_prev"]].round(3).to_string())

rows = []
for a, b in PAIRS:
    f = pairs[f"{a}->{b}"]
    per_day = ((f.imp_next / DAYS_IN_MONTH[b]) < (1 - DECLINE_DROP) * (f.imp / DAYS_IN_MONTH[a])).mean()
    rows.append({"decision_row": f"{a}->{b}", "days": f"{DAYS_IN_MONTH[a]}->{DAYS_IN_MONTH[b]}",
                 "calendar_factor": DAYS_IN_MONTH[b] / DAYS_IN_MONTH[a],
                 "median_page_next_over_now": (f.imp_next / f.imp).median(),
                 "share_of_pages_that_grew": (f.imp_next > f.imp).mean(),
                 "base_rate_as_contracted": f.label_declining.mean(),
                 "base_rate_per_day": per_day,
                 "calendar_shift_pp": (per_day - f.label_declining.mean()) * 100})
drift = pd.DataFrame(rows)
print("\nthe same label definition, four decision months, and what month length alone is worth")
print(drift.round(4).to_string(index=False))
agg_change = tide.imp_per_day_vs_prev.dropna().to_numpy()
print(f"\nwhat tracks the base rate?  corr with the portfolio aggregate per day: "
      f"{np.corrcoef(agg_change, drift.base_rate_as_contracted)[0,1]:+.3f}")
print(f"                            corr with the median page's own ratio:  "
      f"{drift.base_rate_as_contracted.corr(drift.median_page_next_over_now):+.3f}")
print(f"\nbase rate range across the four months: {drift.base_rate_as_contracted.min():.1%} to "
      f"{drift.base_rate_as_contracted.max():.1%}, a {(drift.base_rate_as_contracted.max() - drift.base_rate_as_contracted.min())*100:.0f} "
      f"point swing in the thing being predicted, with the contract unchanged.")

receipts["base_rate_by_decision_month"] = {r.decision_row: round(r.base_rate_as_contracted, 4)
                                           for r in drift.itertuples()}
receipts["calendar_shift_pp"] = {r.decision_row: round(r.calendar_shift_pp, 2) for r in drift.itertuples()}

the panel's own tide: every eligible client, every page, no model in sight
          pages  imp_per_day  imp_per_day_vs_prev
month                                            
2026-01  120932      4655108                  NaN
2026-02  133542      6139839                1.319
2026-03  145238      7787196                1.268
2026-04  151765      7891596                1.013
2026-05  177146      6604161                0.837

the same label definition, four decision months, and what month length alone is worth
    decision_row   days  calendar_factor  median_page_next_over_now  share_of_pages_that_grew  base_rate_as_contracted  base_rate_per_day  calendar_shift_pp
2026-01->2026-02 31->28           0.9032                     1.1385                    0.5967                   0.2512             0.1996            -5.1683
2026-02->2026-03 28->31           1.1071                     1.1875                    0.6360                   0.2290             0.2829             5.3928
2026-03->2026-04 

**The base rate is a property of the month, not of the model.**

The identical contract produces base rates of 25.1%, 22.9%, 53.8% and 53.8% across the four
decision months. The label did not change; the portfolio did.

What tracks the base rate is the *typical* page, and it tracks it almost exactly, at a correlation
of -1.00. The median page reached 1.14x and 1.19x of its own previous month in the first two
windows and 0.76x in both later ones, while the share of pages that grew fell from 64% to 31%. When
most pages are rising, few clear a 20% drop. When most are falling, half of them do.

The portfolio aggregate tells a weaker story, and it is worth stating because it was my first
explanation. Impressions per day across every eligible client rose 31.9% into February and 26.8%
into March, then went flat at 1.3% into April before falling 16.3% into May. That flat April sits
against a 53.8% base rate and a median page down 24%, so the aggregate held up while the typical
page fell. The totals are carried by a handful of large pages and do not describe what happens to
the page an editor is looking at, which is why the median is the honest summary here.

Two consequences follow for every claim in this lane.

- **Precision@50 is not comparable across months.** Week 5's 90.0% sits against a 49.4% base rate,
  while the same queue in a February-style month is scored against 19% to 23%. A queue that looks
  strong in one month and weak in another may be the same queue meeting a different month.
- **Part of the label is the calendar.** The contract compares whole-month impression totals, so a
  28-day month next to a 31-day month moves the threshold by roughly 10% before any content
  changes. Restating the same label on impressions per day, measured in the cell above, moves the
  base rate by -5.2, +5.4, -2.2 and +2.1 points, always in the direction the month lengths predict.
  That is small next to the 31-point tide, but it is real and it is avoidable: w03 chose calendar
  months, and a per-day or trailing-28-day definition would remove it. Changing the contract
  mid-audit would break the comparison, so the effect is recorded here as a known limitation.

In [6]:
def deploy(train_frame, test_frame, restrict_train=None, restrict_test=None):
    """Fit the frozen rule and the chosen forest on one frame, score another. No retuning."""
    A = train_frame if restrict_train is None else train_frame[train_frame.client_hash_id.isin(restrict_train)]
    B = test_frame if restrict_test is None else test_frame[test_frame.client_hash_id.isin(restrict_test)]
    A, B = A.copy(), B.copy()
    med = band_median_ctr(A)
    A["action_score"], B["action_score"] = score_pages(A, med), score_pages(B, med)
    B["model"] = make_forest().fit(A[FEATURES], A.label_declining).predict_proba(B[FEATURES])[:, 1]
    return A, B


def rung(protocol, trained_on, judged_on, A, B):
    return {"protocol": protocol, "trained_on": trained_on, "judged_on": judged_on,
            "pages": len(B), "clients": B.client_hash_id.nunique(),
            "clients_seen_in_training": len(set(A.client_hash_id) & set(B.client_hash_id)),
            "train_base_rate": A.label_declining.mean(), "base_rate": B.label_declining.mean(),
            "rule_p50": p50(B, "action_score"), "model_p50": p50(B, "model"),
            "model_minus_rule": p50(B, "model") - p50(B, "action_score"),
            "model_auc": roc_auc_score(B.label_declining, B.model)}


ladder = []

# Rung A: random rows, one month pair. Both cuts stay INSIDE the training clients, exactly as
# w05 section 2 did, so the held-out clients are not spent on a demonstration.
a_idx, b_idx = next(ShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(train))
A, B = deploy(train.iloc[a_idx], train.iloc[b_idx])
ladder.append(rung("A. random rows, one month pair", "2026-03->04", "2026-03->04", A, B))
assert round(ladder[-1]["model_p50"], 4) == w05["split_contrast"]["random rows (the easy one)"]["precision_at_50"]
assert round(ladder[-1]["model_auc"], 4) == w05["split_contrast"]["random rows (the easy one)"]["auc"]

# Rung B: Week 5's headline: grouped by client, still one month pair, on the sealed clients.
ladder.append(rung("B. grouped by client, one month pair", "2026-03->04", "2026-03->04", train, test))

# Rung C: time-forward, every eligible client. Honest about months, silent about new clients.
for i in range(len(PAIRS) - 1):
    ktr, kte = f"{PAIRS[i][0]}->{PAIRS[i][1]}", f"{PAIRS[i+1][0]}->{PAIRS[i+1][1]}"
    A, B = deploy(pairs[ktr], pairs[kte])
    ladder.append(rung("C. time-forward, all clients", ktr, kte, A, B))

# Rung D: time-forward AND held-out clients: trained on the 18 w05 training clients in month t,
# judged on the 9 w05 held-out clients in month t+1. Both constraints at once.
for i in range(len(PAIRS) - 1):
    ktr, kte = f"{PAIRS[i][0]}->{PAIRS[i][1]}", f"{PAIRS[i+1][0]}->{PAIRS[i+1][1]}"
    A, B = deploy(pairs[ktr], pairs[kte], TRAIN_CLIENTS, TEST_CLIENTS)
    ladder.append(rung("D. time-forward + held-out clients", ktr, kte, A, B))

ladder = pd.DataFrame(ladder)
print("THE LADDER: same five features, same rule, same forest, same metric. Only the split changes.")
print(ladder.round(4).to_string(index=False))

D = ladder[ladder.protocol.str.startswith("D.")]
print(f"\nrung D, the honest one: model minus rule = "
      f"{', '.join(f'{v:+.0%}' for v in D.model_minus_rule)} across the three deployments "
      f"(mean {D.model_minus_rule.mean():+.1%})")

receipts["ladder"] = {f"{r.protocol} | {r.judged_on}": {
    "rule_p50": round(r.rule_p50, 4), "model_p50": round(r.model_p50, 4),
    "base_rate": round(r.base_rate, 4), "model_auc": round(r.model_auc, 4)} for r in ladder.itertuples()}

THE LADDER: same five features, same rule, same forest, same metric. Only the split changes.
                            protocol       trained_on        judged_on  pages  clients  clients_seen_in_training  train_base_rate  base_rate  rule_p50  model_p50  model_minus_rule  model_auc
      A. random rows, one month pair      2026-03->04      2026-03->04  17905       17                        17           0.5567     0.5572      0.98       1.00              0.02     0.7664
B. grouped by client, one month pair      2026-03->04      2026-03->04  25771        9                         0           0.5569     0.4936      0.90       0.90              0.00     0.6532
        C. time-forward, all clients 2026-01->2026-02 2026-02->2026-03  72745       28                        28           0.2512     0.2290      0.96       0.94             -0.02     0.6971
        C. time-forward, all clients 2026-02->2026-03 2026-03->2026-04  85453       27                        25           0.2290     0.5378   

In [7]:
# Why rung D is as strict as this panel allows: the "new client next month" test cannot be run.
print("clients arriving between adjacent decision months (eligible clients only):")
for i in range(len(PAIRS) - 1):
    ktr, kte = f"{PAIRS[i][0]}->{PAIRS[i][1]}", f"{PAIRS[i+1][0]}->{PAIRS[i+1][1]}"
    seen = set(pairs[ktr].client_hash_id)
    fresh = pairs[kte][~pairs[kte].client_hash_id.isin(seen)]
    print(f"  trained {ktr} -> deployed {kte}: {fresh.client_hash_id.nunique()} unseen clients, "
          f"{len(fresh):,} pages (need {TOP_K} for a queue)")
print("\nSo 'a new client, next month' is not measurable here: the eligible panel is a fixed set of")
print("clients with long GSC history. Rung D approximates it by holding out clients across time")
print("instead, the model trains on 18 clients in one month and is judged on 9 different clients")
print("in the next. The genuinely new-client question stays with the grouped split, in one month.")

clients arriving between adjacent decision months (eligible clients only):
  trained 2026-01->2026-02 -> deployed 2026-02->2026-03: 0 unseen clients, 0 pages (need 50 for a queue)
  trained 2026-02->2026-03 -> deployed 2026-03->2026-04: 2 unseen clients, 3 pages (need 50 for a queue)
  trained 2026-03->2026-04 -> deployed 2026-04->2026-05: 0 unseen clients, 0 pages (need 50 for a queue)

So 'a new client, next month' is not measurable here: the eligible panel is a fixed set of
clients with long GSC history. Rung D approximates it by holding out clients across time
instead, the model trains on 18 clients in one month and is judged on 9 different clients
in the next. The genuinely new-client question stays with the grouped split, in one month.


**Reading the ladder: the easy split was worth ten points, and time costs the model its edge.**

- **Rungs A and B, the split Week 5 already reported.** Random rows give precision@50 of 100.0% and
  AUC 0.766. Grouped by client gives 90.0% and AUC 0.653. Same model, same rows, ten points and a
  perfect fifty out of fifty, bought entirely by letting the model meet its own clients again. Both
  are recomputed here and asserted against the w05 receipt.
- **Rung C, forward in time across all clients.** The rule scores 0.96, 0.92 and 0.96; the model
  scores 0.94, 0.86 and 0.98. These look *better* than Week 5's 90%, which is the trap this rung
  exists to expose. It scores clients the model already trained on in an earlier month, so it has
  relaxed the client constraint while tightening the time one. Read on its own it would make a
  flattering headline.
- **Rung D, both constraints at once.** Model minus rule comes to -10, +8 and -2 points across the
  three deployments, a mean of -1.3. The frozen rule scores 0.88, 0.90 and 0.96; the model scores
  0.78, 0.98 and 0.94.

Week 5's verdict survives and gets slightly worse. The tie held across 25 re-cuts of one month
pair. Across three deployments the model trails by an average of 1.3 points, and the difference
itself swings 18 points between its best and worst month. Section 4 puts an interval on that rather than
quoting whichever end flatters the model.

Two further things rung D shows that the grouped split could not.

- **The model's AUC in the first deployment is 0.581**, barely above coin-flipping, on the same
  five features that scored 0.653 in Week 5. Nothing about the model changed. It was trained on
  January to February and deployed into February to March.
- **The training month's base rate does not match the deployment month's.** The middle deployment
  trains where 18.9% of pages decline and is judged where 49.4% do. A forest calibrated on the
  first cannot output honest probabilities for the second. That is the second reason, after Week
  5's calibration table, why this ships as an ordered queue and never as a percentage.

### 2c. Two features that change shape with the calendar

Section 2 compares months, so a feature that behaves differently in a 28-day month than in a
31-day one can move the verdict on its own. Two of the five do.

`momentum_in_month` divides the month at a fixed day. w03 used the 17th of March, which splits 31
days into 16 early and 15 late, so the late half is one day shorter and the ratio sits about 6%
low. February divides 14/14 and April 15/15, both symmetric. Extending w03's constant across the
calendar produced three different splits rather than one.

`active_days` counts days with impressions, so its ceiling is 28 in February and 31 in March.

The cell below measures both, then rebuilds rung D twice: once as contracted, once with the two
features expressed per day.

In [8]:
EARLY_DAYS = {m: int(MID[m][-2:]) - 1 for m in MONTHS}      # days before the mid-month cut

print("the mid-month split is not the same shape in every month:")
for m in MONTHS:
    early, late = EARLY_DAYS[m], DAYS_IN_MONTH[m] - EARLY_DAYS[m]
    slice_m = panel[(panel.month == m) & panel.client_hash_id.isin(eligible_clients)
                    & (panel.imp >= MIN_IMP)]
    mom = slice_m.imp_late / slice_m.imp_early.replace(0, np.nan)
    print(f"  {m}: {early}d early / {late}d late (late half is {late/early:.3f} of early) | "
          f"median momentum {mom.median():.3f} | per-day median {(mom * early / late).median():.3f} | "
          f"active_days at the {DAYS_IN_MONTH[m]}-day cap for "
          f"{(slice_m.active_days == DAYS_IN_MONTH[m]).mean():.1%} of pages")


def build_per_day(feature_month, label_month):
    """The same rows, with the two calendar-shaped features expressed per day."""
    fr = build_pair(feature_month, label_month)
    early = EARLY_DAYS[feature_month]
    late = DAYS_IN_MONTH[feature_month] - early
    fr["momentum_in_month"] = ((fr.imp_late / late) / (fr.imp_early / early).replace(0, np.nan)).fillna(1.0)
    fr["active_days"] = fr.active_days / DAYS_IN_MONTH[feature_month]
    return fr


sensitivity = []
for tag, builder in [("as contracted (w03 calendar months)", build_pair),
                     ("momentum and active_days per day", build_per_day)]:
    for i in range(len(PAIRS) - 1):
        ktr, kte = f"{PAIRS[i][0]}->{PAIRS[i][1]}", f"{PAIRS[i+1][0]}->{PAIRS[i+1][1]}"
        A, B = deploy(builder(*PAIRS[i]), builder(*PAIRS[i+1]), TRAIN_CLIENTS, TEST_CLIENTS)
        sensitivity.append({"features": tag, "judged_on": kte, "rule_p50": p50(B, "action_score"),
                            "model_p50": p50(B, "model"),
                            "model_minus_rule": p50(B, "model") - p50(B, "action_score"),
                            "model_auc": roc_auc_score(B.label_declining, B.model)})

sensitivity = pd.DataFrame(sensitivity)
print("\nrung D rebuilt both ways (trained on 18 clients in month t, judged on 9 clients in t+1):")
print(sensitivity.round(4).to_string(index=False))
for tag in sensitivity.features.unique():
    d = sensitivity[sensitivity.features == tag].model_minus_rule
    print(f"  {tag:<38} mean model minus rule {d.mean():+.4f}")

receipts["calendar_sensitivity"] = {
    tag: {"per_deployment": [round(v, 4) for v in sensitivity[sensitivity.features == tag].model_minus_rule],
          "mean": round(float(sensitivity[sensitivity.features == tag].model_minus_rule.mean()), 4)}
    for tag in sensitivity.features.unique()}

the mid-month split is not the same shape in every month:
  2026-01: 16d early / 15d late (late half is 0.938 of early) | median momentum 1.177 | per-day median 1.256 | active_days at the 31-day cap for 74.1% of pages
  2026-02: 14d early / 14d late (late half is 1.000 of early) | median momentum 1.127 | per-day median 1.127 | active_days at the 28-day cap for 76.2% of pages


  2026-03: 16d early / 15d late (late half is 0.938 of early) | median momentum 0.991 | per-day median 1.057 | active_days at the 31-day cap for 62.9% of pages
  2026-04: 15d early / 15d late (late half is 1.000 of early) | median momentum 0.829 | per-day median 0.829 | active_days at the 30-day cap for 72.6% of pages
  2026-05: 16d early / 15d late (late half is 0.938 of early) | median momentum 0.911 | per-day median 0.972 | active_days at the 31-day cap for 73.4% of pages



rung D rebuilt both ways (trained on 18 clients in month t, judged on 9 clients in t+1):
                           features        judged_on  rule_p50  model_p50  model_minus_rule  model_auc
as contracted (w03 calendar months) 2026-02->2026-03      0.88       0.78             -0.10     0.5807
as contracted (w03 calendar months) 2026-03->2026-04      0.90       0.98              0.08     0.6723
as contracted (w03 calendar months) 2026-04->2026-05      0.96       0.94             -0.02     0.7162
   momentum and active_days per day 2026-02->2026-03      0.88       0.72             -0.16     0.5908
   momentum and active_days per day 2026-03->2026-04      0.90       0.98              0.08     0.6720
   momentum and active_days per day 2026-04->2026-05      0.96       0.96              0.00     0.7165
  as contracted (w03 calendar months)    mean model minus rule -0.0133
  momentum and active_days per day       mean model minus rule -0.0267


**The calendar is visible in both features, and the verdict does not move.**

In the three 31-day months the late half is 0.938 of the early half, so `momentum_in_month` reads
about 6% below what a symmetric split would give, while February and April are clean. `active_days`
turns out to be saturated rather than informative: between 63% and 76% of pages sit at the month's
ceiling, which is the simplest explanation for Week 5 finding that shuffling the column slightly
*improved* precision@50.

Rebuilt with both features expressed per day, rung D gives model minus rule of -16, +8 and 0
points, a mean of -2.7, against -10, +8 and -2 for a mean of -1.3 as contracted. The deficit holds
and widens slightly. That is the useful direction: had the per-day version turned the deficit into
a lead, rung D would have been measuring the calendar handling rather than the model.

This is a sensitivity check, so every other number in the notebook stays on the contracted
definition. The fix belongs in the contract, beside the per-day label from 2a.

In [9]:
# The deployment that failed, up close. Trained Jan->Feb on the 18 training clients,
# deployed Feb->Mar on the 9 held-out clients. Only decision-time columns are shown, plus what
# the next month actually did, printed after the judgement, never before it.
A_fail, B_fail = deploy(pairs["2026-01->2026-02"], pairs["2026-02->2026-03"], TRAIN_CLIENTS, TEST_CLIENTS)
print(f"trained on 2026-01->02 ({len(A_fail):,} pages, base {A_fail.label_declining.mean():.1%}) | "
      f"deployed on 2026-02->03 ({len(B_fail):,} pages / {B_fail.client_hash_id.nunique()} clients, "
      f"base {B_fail.label_declining.mean():.1%})")
print(f"rule precision@{TOP_K} {p50(B_fail,'action_score'):.0%} | model {p50(B_fail,'model'):.0%} | "
      f"model AUC {roc_auc_score(B_fail.label_declining, B_fail.model):.4f}")

top = B_fail.nlargest(TOP_K, "model")
misses = top[top.label_declining == 0]
show = (misses.assign(next_over_now=(misses.imp_next / misses.imp).round(2),
                      ctr_pct=(misses.ctr_mar * 100).round(3))
        [["imp_mar", "momentum_in_month", "pos_mar", "ctr_pct", "active_days", "spike_share",
          "next_over_now"]].round(3))
print(f"\nthe model's {len(misses)} wrong picks out of its top {TOP_K} (client ids withheld):")
print(show.to_string(index=False))
print(f"\nwrong picks : median {misses.imp_mar.median():,.0f} impressions | median in-month momentum "
      f"{misses.momentum_in_month.median():.2f} | median next/now {(misses.imp_next/misses.imp).median():.2f}")
right = top[top.label_declining == 1]
print(f"right picks : median {right.imp_mar.median():,.0f} impressions | median in-month momentum "
      f"{right.momentum_in_month.median():.2f}")

blind_fail = B_fail[B_fail.action_score == 0]
bl_top = blind_fail.nlargest(TOP_K, "model")
print(f"\ninside the rule's blind spot that month: {len(blind_fail):,} pages, base rate "
      f"{blind_fail.label_declining.mean():.1%}, model precision@{TOP_K} there {p50(blind_fail,'model'):.0%}")
print(f"  the 50 pages it picked in there: median {bl_top.imp_mar.median():,.0f} impressions, median "
      f"momentum {bl_top.momentum_in_month.median():.2f}, median next/now "
      f"{(bl_top.imp_next/bl_top.imp).median():.2f}")
print(f"  the median page in that blind spot went to "
      f"{(blind_fail.imp_next/blind_fail.imp).median():.2f}x of its own previous month")

from scipy.stats import mannwhitneyu
hits = top[top.label_declining == 1]
u_stat, p_size = mannwhitneyu(misses.imp_mar, hits.imp_mar)
print(f"\nare the wrong picks a different size from the right ones? "
      f"Mann-Whitney p = {p_size:.2f} on {len(misses)} against {len(hits)} "
      f"-> {'no detectable difference' if p_size > 0.05 else 'they differ'}")
print(f"and the label is a threshold, not a fact: the mildest miss came in at "
      f"{(misses.imp_next/misses.imp).min():.2f}x, so it did lose traffic, just not past the "
      f"{DECLINE_DROP:.0%} line")

receipts["failed_deployment"] = {
    "trained_on": "2026-01->2026-02", "judged_on": "2026-02->2026-03",
    "pages": int(len(B_fail)), "base_rate": round(float(B_fail.label_declining.mean()), 4),
    "rule_p50": round(p50(B_fail, "action_score"), 4), "model_p50": round(p50(B_fail, "model"), 4),
    "model_auc": round(float(roc_auc_score(B_fail.label_declining, B_fail.model)), 4),
    "blind_spot_base_rate": round(float(blind_fail.label_declining.mean()), 4),
    "blind_spot_model_p50": round(p50(blind_fail, "model"), 4),
    "miss_vs_hit_size_mannwhitney_p": round(float(p_size), 4)}

trained on 2026-01->02 (40,972 pages, base 22.6%) | deployed on 2026-02->03 (21,256 pages / 8 clients, base 19.4%)
rule precision@50 88% | model 78% | model AUC 0.5807

the model's 11 wrong picks out of its top 50 (client ids withheld):
 imp_mar  momentum_in_month  pos_mar  ctr_pct  active_days  spike_share  next_over_now
  9233.0              0.380    5.458    0.032           28        0.145           2.54
  8184.0              0.458    2.905    0.086           28        0.186           1.13
 19996.0              0.505    4.726    0.040           28        0.255           1.28
 20397.0              0.550    3.659    0.098           28        0.225           1.13
 27183.0              0.423    4.054    0.110           28        0.223           1.67
   101.0              0.086   69.901    0.000           18        0.347           3.11
   235.0              0.068   83.732    0.000           24        0.255           2.15
   508.0              0.076   70.539    0.000           25        0

**The failure is not new. It is Week 5's error mode meeting a different month.**

Week 5 described the model's wrong picks precisely: pages that slid hard inside the feature month
and then bounced back. All eleven above are that page. Their median in-month momentum is 0.38, so
they were halving inside February, and their median next month came in at 1.28x, not down at all.
The model did not develop a new weakness. It walked into the same one in a month when the whole
panel was rising, so far more of those pages bounced.

What changed is the size of the mistake. Week 5's misses were small pages of 942 to 2,535
impressions. These have a median of 9,233 against 7,427 for its correct picks, and a
Mann-Whitney test on 11 against 39 returns p = 0.42, so no size difference is detectable at this
sample size. The same error now lands on pages where a wasted editorial afternoon costs the most,
which is the failure Week 5's error table by size warned about. One of the eleven is also a
reminder that the label is a threshold rather than a fact: it came in at 0.86x, so it did lose
traffic, just not past the 20% line.

The blind spot is where this stops being a rounding error. That month the model ranked 4% correct
inside the rule's blind spot against a 17.2% base rate there, worse than picking at random from the
same pool. The 50 pages it chose had a median momentum of 1.12, meaning they were *growing*, and
they went on to grow 1.85x. Asked to rank pages that were not sliding, in a month when nothing was
sliding, the model sorted by the wrong end of the axis. That is the finding Week 5 called "the one
job the model does that my rule cannot", and section 4 rewrites it.

## 3. Leakage audit

An answer sneaks into a model in three ways: through a feature computed **from the label**, a
feature drawn from **after the decision moment**, or a feature carrying **another system's
decision**. Week 3 ran this hunt on a candidate feature list. This section runs it on the five
features that actually shipped, plus the population rules that decide which rows exist at all, and
it runs as code rather than from memory.

In [10]:
# --- 3a. the timeline, read from the warehouse rather than from memory -------------------------
window = hf_sql(f"""
    SELECT '{FEATURE_MONTH}' AS month, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(*) AS daily_rows FROM {fact(FEATURE_MONTH)}
    UNION ALL
    SELECT '{LABEL_MONTH}', MIN(report_date), MAX(report_date), COUNT(*) FROM {fact(LABEL_MONTH)}
    ORDER BY 1
""")
print("the two windows, as the warehouse reports them:")
print(window.to_string(index=False))

decision = pd.Timestamp(DECISION_DATE)
feat_last = pd.Timestamp(window.loc[window.month == FEATURE_MONTH, "last_day"].iloc[0])
label_first = pd.Timestamp(window.loc[window.month == LABEL_MONTH, "first_day"].iloc[0])
assert feat_last <= decision, "a feature day falls after the decision moment"
assert label_first > decision, "a label day falls on or before the decision moment"
print(f"\nfeature window ends {feat_last.date()} <= decision {decision.date()} < label window starts "
      f"{label_first.date()}  ->  no overlap, no gap crossed")

# every feature, its source columns, and the window each one is measured over
timeline = pd.DataFrame([
    ("log_imp_mar",       "gsc_impressions",                 FEATURE_MONTH, "whole month"),
    ("active_days",       "gsc_impressions (day count)",     FEATURE_MONTH, "whole month"),
    ("pos_mar",           "gsc_sum_position / gsc_impressions", FEATURE_MONTH, "whole month"),
    ("ctr_mar",           "gsc_clicks / gsc_impressions",    FEATURE_MONTH, "whole month"),
    ("momentum_in_month", "gsc_impressions, split at " + MID[FEATURE_MONTH], FEATURE_MONTH, "two halves"),
    ("label_declining",   "gsc_impressions",                 LABEL_MONTH,   "whole month  <-- the answer"),
], columns=["column", "built_from", "measured_in", "window"])
timeline["knowable_on_" + DECISION_DATE.replace("-", "")] = timeline.measured_in.eq(FEATURE_MONTH)
print("\nevery column the model sees, and when it becomes knowable:")
print(timeline.to_string(index=False))
assert timeline[timeline.column.isin(FEATURES)].iloc[:, -1].all(), "a feature is not knowable at decision time"
assert not timeline.loc[timeline.column == "label_declining"].iloc[:, -1].item(), "the label is not in the future"

# --- decision-derived columns: what exists in the warehouse, and what the contract refuses ------
product_columns = hf_sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT}").column_name.tolist()
decision_derived = [c for c in product_columns
                    if c in {"last_optimized_date", "optimization_eligible_date", "is_published",
                             "is_deleted", "provider_used", "model_used", "competition_level",
                             "main_intent"}]
print(f"\ndecision-derived columns available in dim_content: {', '.join(decision_derived)}")
print(f"used as features: {[c for c in decision_derived if c in FEATURES]}  <- must be empty")
assert not any(c in FEATURES for c in product_columns), "a dim_content column reached the feature list"
assert not any(c in FEATURES for c in ("client_hash_id", "content_hash_id", "url_hash_id",
                                       "keyword_hash_id")), "an ID reached the feature list"
assert "action_score" not in FEATURES, "the w04 rule's own score reached the feature list"
print("IDs used for grouping and joining only. w04's action_score is the baseline to beat, never an input.")
print(f"tables touched by the whole pipeline: fact_content_daily_performance (gsc_* columns only), "
      f"dim_clients (gsc_data_start only). dim_content is read in 3c to AUDIT the label, never to build it.")

the two windows, as the warehouse reports them:
  month  first_day   last_day  daily_rows
2026-03 2026-03-01 2026-03-31     9841378
2026-04 2026-04-01 2026-04-30    10424730

feature window ends 2026-03-31 <= decision 2026-03-31 < label window starts 2026-04-01  ->  no overlap, no gap crossed

every column the model sees, and when it becomes knowable:
           column                           built_from measured_in                      window  knowable_on_20260331
      log_imp_mar                      gsc_impressions     2026-03                 whole month                  True
      active_days          gsc_impressions (day count)     2026-03                 whole month                  True
          pos_mar   gsc_sum_position / gsc_impressions     2026-03                 whole month                  True
          ctr_mar         gsc_clicks / gsc_impressions     2026-03                 whole month                  True
momentum_in_month gsc_impressions, split at 2026-03-17     20


decision-derived columns available in dim_content: competition_level, main_intent, provider_used, model_used, last_optimized_date, optimization_eligible_date, is_published, is_deleted
used as features: []  <- must be empty
IDs used for grouping and joining only. w04's action_score is the baseline to beat, never an input.
tables touched by the whole pipeline: fact_content_daily_performance (gsc_* columns only), dim_clients (gsc_data_start only). dim_content is read in 3c to AUDIT the label, never to build it.


**The timeline is clean, and it is the easy half of the audit.** Every feature is built from
`gsc_*` columns inside 2026-03, which closes on the decision date, and the label is built from
2026-04, which opens the day after. No 90-day aggregate straddles the boundary, because the
contract never used one. The five features touch two tables and eight columns. `dim_content`, where
the product's own decisions live in `last_optimized_date`, `is_deleted` and
`optimization_eligible_date`, is not in the pipeline at all.

The harder half is whether the harness could see a leak if one were there, so the next cell plants
two on purpose.

In [11]:
# --- 3b. plant a leak, watch it explode; then ablate the honest features -----------------------
train_x, test_x = train.copy(), test.copy()
for d in (train_x, test_x):
    d["LEAK_next_month_impressions"] = np.log1p(d.imp_next)   # a column from the label window
    d["LEAK_the_label_itself"] = d.label_declining            # the answer, spelled out


def fit_and_score(feature_list, tr=train_x, te=test_x):
    m = make_forest().fit(tr[feature_list], tr.label_declining)
    risk = m.predict_proba(te[feature_list])[:, 1]
    return p50(te.assign(_r=risk), "_r"), roc_auc_score(te.label_declining, risk)


honest = fit_and_score(FEATURES)
print(f"{'honest: the 5 contract features':<48} precision@50 {honest[0]:.3f}   AUC {honest[1]:.4f}")
leak_rows = {}
for extra in ("LEAK_next_month_impressions", "LEAK_the_label_itself"):
    r = fit_and_score(FEATURES + [extra])
    leak_rows[extra] = r
    print(f"{'  + ' + extra:<48} precision@50 {r[0]:.3f}   AUC {r[1]:.4f}")
assert leak_rows["LEAK_the_label_itself"][1] > 0.99, "the harness cannot detect a planted label, so the test is broken"
print("\nThe harness works: handed the answer, the same pipeline prints a perfect score. So when the")
print("honest features print 0.65 AUC, that is the data being hard, not the test being blind.")

print("\nleave-one-out on the honest five (a leak collapses when removed; a signal only sags):")
abl = []
for f in FEATURES:
    r = fit_and_score([c for c in FEATURES if c != f])
    abl.append({"removed": f, "precision_at_50": r[0], "auc": r[1],
                "auc_drop_vs_honest": honest[1] - r[1]})
solo = fit_and_score(["momentum_in_month"])
abl.append({"removed": "everything except momentum_in_month", "precision_at_50": solo[0],
            "auc": solo[1], "auc_drop_vs_honest": honest[1] - solo[1]})
print(pd.DataFrame(abl).round(4).to_string(index=False))

receipts["leak_probe"] = {"honest_p50": round(honest[0], 4), "honest_auc": round(honest[1], 4),
                          **{k: {"p50": round(v[0], 4), "auc": round(v[1], 4)} for k, v in leak_rows.items()}}
receipts["ablation_auc"] = {r["removed"]: round(r["auc"], 4) for r in abl}

honest: the 5 contract features                  precision@50 0.900   AUC 0.6532


  + LEAK_next_month_impressions                  precision@50 1.000   AUC 0.9257


  + LEAK_the_label_itself                        precision@50 1.000   AUC 1.0000

The harness works: handed the answer, the same pipeline prints a perfect score. So when the
honest features print 0.65 AUC, that is the data being hard, not the test being blind.

leave-one-out on the honest five (a leak collapses when removed; a signal only sags):


                            removed  precision_at_50    auc  auc_drop_vs_honest
                        log_imp_mar             0.88 0.6496              0.0036
                        active_days             0.92 0.6607             -0.0075
                            pos_mar             0.96 0.6493              0.0039
                            ctr_mar             0.90 0.6452              0.0080
                  momentum_in_month             0.82 0.5912              0.0620
everything except momentum_in_month             0.90 0.6428              0.0104


**No leak signature, and the ablation says something else worth knowing.**

Planting next month's impressions moves AUC from 0.653 to 0.926. Planting the label itself moves it
to 1.000 and precision@50 to a perfect fifty out of fifty. That is what a leak looks like in this
harness, and it matches the deliberate demonstration w03 committed, where `leaked_score` reached
AUC 0.9995. Against that, the honest features look nothing like a leak.

The leave-one-out table then makes Week 5's uncomfortable finding harder to avoid. Removing
`momentum_in_month` costs 0.062 AUC, from 0.653 down to 0.591, the largest drop and still not a
collapse. Removing any of the other four costs 0.008 or less, removing `active_days` slightly
*improves* AUC, and dropping `pos_mar` lifts precision@50 to 0.960. The last row is the blunt one:
`momentum_in_month` alone scores precision@50 of 0.900 and AUC 0.643, the same queue precision as
the full model from a single column.

So this is a five-feature model in name and close to a one-feature model in behaviour, exactly as
Week 5's permutation importances said. That is not leakage. It is a limit on what "the model
learned five things" can be taken to mean.

In [12]:
# --- 3c. the population: which rows exist, and does any of that use the outcome window? --------
missing = frame[frame.seen_next_month == 0]
print(f"pages with no {LABEL_MONTH} row at all: {len(missing)} of {len(frame):,} "
      f"({len(missing)/len(frame)*100:.2f}% of the slice)")
print(f"  they are scored as imp_next = 0, so all {len(missing)} are labelled declining")
print(f"  they make up {missing.label_declining.sum()/frame.label_declining.sum()*100:.2f}% of every positive")
print(f"  base rate with them {frame.label_declining.mean():.4f} | without them "
      f"{frame[frame.seen_next_month == 1].label_declining.mean():.4f}")
kept = test[test.seen_next_month == 1]
print(f"  held-out precision@{TOP_K} without them: rule {p50(kept,'action_score'):.3f} "
      f"(vs {p50(test,'action_score'):.3f}), model {p50(kept,'model'):.3f} (vs {p50(test,'model'):.3f})")

# Is "no next-month row" an organic disappearance, or an editorial decision? dim_content is read
# HERE to audit the label. It is a snapshot taken after the fact, so it may never be a feature.
audit_cols = hf_sql(f"SELECT content_hash_id, is_deleted, is_published FROM {DIM_CONTENT}")
j = frame.merge(audit_cols, on="content_hash_id", how="left")
print("\nwhat dim_content says about those pages (audit only: these columns are never features):")
print(j.groupby("seen_next_month").agg(pages=("content_hash_id", "size"),
                                       share_is_deleted=("is_deleted", "mean"),
                                       share_is_published=("is_published", "mean"),
                                       decline_rate=("label_declining", "mean")).round(4).to_string())
print(f"\n  deleted pages are {j[j.is_deleted == True].shape[0]} of {len(j):,} rows and decline at "
      f"{j[j.is_deleted == True].label_declining.mean():.1%} vs {j[j.is_deleted != True].label_declining.mean():.1%}")

# the client side of the population, a survivor set by construction
n_clients_total = int(hf_sql(f"SELECT COUNT(*) AS n FROM {DIM_CLIENTS}").n.iloc[0])
print(f"\nclients: {n_clients_total} in the warehouse -> {len(eligible_clients)} with GSC history from "
      f"before {MIN_HISTORY} -> {frame.client_hash_id.nunique()} with pages clearing {MIN_IMP} impressions")

# the label's denominator is also one of the features
frame["size_decile"] = pd.qcut(frame.imp_mar, 10, labels=False, duplicates="drop")
by_size = frame.groupby("size_decile").agg(pages=("imp_mar", "size"), median_impressions=("imp_mar", "median"),
                                           decline_rate=("label_declining", "mean"))
print("\nthe label is a ratio to the feature month, so it is not size-neutral:")
print(by_size.round(3).to_string())
print(f"  smallest decile declines at {by_size.decline_rate.iloc[0]:.1%}, largest at "
      f"{by_size.decline_rate.iloc[-1]:.1%}, a {(by_size.decline_rate.iloc[0]-by_size.decline_rate.iloc[-1])*100:.1f} "
      f"point spread built into the target itself")

receipts["population_audit"] = {
    "pages_no_next_month_row": int(len(missing)),
    "share_of_positives_pct": round(float(missing.label_declining.sum()/frame.label_declining.sum()*100), 2),
    "base_rate_excluding_them": round(float(frame[frame.seen_next_month == 1].label_declining.mean()), 4),
    "clients_in_warehouse": n_clients_total, "clients_eligible": len(eligible_clients),
    "clients_in_slice": int(frame.client_hash_id.nunique()),
    "decline_rate_smallest_decile": round(float(by_size.decline_rate.iloc[0]), 4),
    "decline_rate_largest_decile": round(float(by_size.decline_rate.iloc[-1]), 4)}

pages with no 2026-04 row at all: 380 of 85,453 (0.44% of the slice)
  they are scored as imp_next = 0, so all 380 are labelled declining
  they make up 0.83% of every positive
  base rate with them 0.5378 | without them 0.5357
  held-out precision@50 without them: rule 0.900 (vs 0.900), model 0.900 (vs 0.900)



what dim_content says about those pages (audit only: these columns are never features):
                 pages  share_is_deleted  share_is_published  decline_rate
seen_next_month                                                           
0                  380            0.0289              0.9658        1.0000
1                85073            0.0002              0.9998        0.5357

  deleted pages are 27 of 85,453 rows and decline at 88.9% vs 53.8%



clients: 104 in the warehouse -> 40 with GSC history from before 2026-01-01 -> 27 with pages clearing 100 impressions

the label is a ratio to the feature month, so it is not size-neutral:
             pages  median_impressions  decline_rate
size_decile                                         
0             8643               125.0         0.547
1             8461               191.0         0.554
2             8575               287.0         0.564
3             8526               433.0         0.570
4             8528               651.0         0.556
5             8546               999.0         0.565
6             8550              1575.0         0.549
7             8538              2598.0         0.531
8             8540              4802.0         0.467
9             8546             11323.0         0.476
  smallest decile declines at 54.7%, largest at 47.6%, a 7.1 point spread built into the target itself


**Where the population is clean, where it is a choice, and where it is a survivor set.**

- **Row selection never touches the outcome window.** A page is in the slice because of what it did
  in March: data available, at least 100 impressions, and a client with history. Nothing conditions
  on April. That is the check Finding #2 in the paper could not pass, and this lane passes it by
  construction rather than by luck.
- **380 pages, 0.44% of the slice, stop appearing in April** and are counted as total losses. They
  make up 0.83% of all positives. Removing them moves the base rate from 0.5378 to 0.5357, and
  moves the held-out precision@50 of both scorers by nothing at all. Small, disclosed, measured.
- **They are mostly not deletions.** `is_deleted` is true for 11 of the 380 disappearing pages
  and for 16 of the other 85,073, which is a 154x concentration and still only eleven pages. So
  vanishing from Search Console is overwhelmingly organic in this slice rather than a CMS action.
  (`dim_content` is a snapshot taken after the fact. It is read here to audit the label, and is
  disqualified as a feature for exactly that reason.)
- **The clients are a survivor set, and this is the real limitation.** Of 104 clients in the
  warehouse, 40 have GSC history reaching back before 2026-01-01, and 27 contribute a page clearing
  100 impressions. Every number in this lane describes established accounts with long history. Section 2's
  unseen-client check found the consequence: across adjacent months there is no new client to test
  on, because the panel does not contain one.
- **The label is not size-neutral.** Its denominator is the feature month, and `log_imp_mar` is one
  of the five features. Small pages decline at 54.7% and the largest decile at 47.6%, a 7.1 point
  spread built into the target. This is not leakage, since March is knowable on 31 March, but it
  explains why a queue optimised for precision drifts toward small pages, and why Week 5's rule
  protected six times more traffic at identical precision.

### The attack checklist, with a verdict on each line

| Check (from `hunting-leakage-and-validating`) | Verdict | The number behind it |
|---|---|---|
| Timeline drawn: features strictly before the label window | **pass** | feature window ends 2026-03-31, label opens 2026-04-01, asserted in 3a |
| No label-derived or sibling columns among the features | **pass** | leave-one-out costs ≤0.062 AUC, while a planted label *gains* 0.347 |
| No product flags or existing-system scores as features | **pass** | `dim_content` never read in the build; `action_score` is the baseline, asserted in 3a |
| Population selection checked for outcome-window information | **pass, disclosed** | selection uses March only; 380 pages, 0.44%, disappear next month and are kept as declines |
| Split grouped by the repeating entity and/or time-based | **pass, both** | grouped by client in section 0c, and forward across 3 deployments in section 2 |
| Base rate printed next to every metric | **pass** | and it moves from 22.9% to 53.8% across months, per section 2 |
| Top feature importance sanity-checked | **pass, with a caveat** | one column carries the queue: momentum alone reaches 0.900 precision@50 |
| Metrics recomputed out-of-fold, never in-sample | **pass** | every score in section 2 comes from clients or months outside its own training frame |
| Sealed or holdout claims leave receipts | **pass** | Section 0c asserts nine committed numbers, and this notebook writes its own receipts in section 4 |

One line cannot be marked clean, and it is a limitation rather than a leak. June 2026 stays sealed,
so the strongest test available, deploying into a month nobody has looked at, remains unspent. That
is deliberate, and it is what the capstone gets to use.

## 4. Claim rewrite

Week 5's boldest sentence is not the headline, because the headline was already a tie and already
honest. It is the sentence in section 4 that survived a check and was promoted to a finding:

> **As written in w05:** "Ranked *inside* that blind spot, the chosen forest reaches **70.0%
> precision@50** against a **40.7%** base rate there — 35 correct picks in fifty where picking at
> random inside the same pool gets 20. And unlike every other margin in this notebook, **this one
> reproduces**: the same forest averages **71.2%** in the blind spot across grouped CV on the
> *training* clients... So the model can see something real in pages the rule declares
> uninteresting."

Four things are wrong with it, and only one of them is the number.

1. **"Reproduces" was proven in the wrong dimension.** Both checks, held-out clients and
   training-client CV, sit inside the same March to April transition. That establishes the effect
   is not one client's quirk. It says nothing about a different month, which is where section 2 found the
   exposure.
2. **A single point estimate is quoted to one decimal.** 70.0% is 35 pages out of 50.
3. **"Can see something real" reaches for a property of the world.** What was observed is a ranking
   that beat a base rate on one month of pages.
4. **No uncertainty is attached to a claim built on 50 items.**

The cell below measures all four before the sentence is rewritten.

In [13]:
# --- 4. how much uncertainty is around the numbers I want to claim? ---------------------------
def p50_fast(scores, labels, k=TOP_K):
    """precision_at_k_fair on raw arrays, same maths, fast enough to bootstrap."""
    cut = np.partition(scores, -k)[-k]
    above, tied = scores > cut, scores == cut
    slots = k - int(above.sum())
    return (labels[above].sum() + slots * labels[tied].mean()) / k


assert abs(p50_fast(test.model.to_numpy(), test.label_declining.to_numpy()) - p50(test, "model")) < 1e-12


def wilson(hits, n, z=1.645):          # 90% interval
    p = hits / n
    denom = 1 + z*z/n
    centre = p + z*z/(2*n)
    half = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (centre - half)/denom, (centre + half)/denom


lo, hi = wilson(round(W05_MODEL_P50 * TOP_K), TOP_K)
print(f"1) a queue of {TOP_K} is a small sample. {round(W05_MODEL_P50*TOP_K)} correct out of {TOP_K} "
      f"= {W05_MODEL_P50:.1%}, Wilson 90% interval [{lo:.1%}, {hi:.1%}]")
print(f"   the rule and the model are {abs(W05_MODEL_P50-W05_RULE_P50)*100:.0f} points apart inside "
      f"a {((hi-lo)*100):.0f}-point interval.")

# --- cluster bootstrap by client, inside Week 5's own held-out month ---
rng = np.random.default_rng(SEED)
B_ITERS = 2000


ids = test.client_hash_id.unique()
blocks = [test[test.client_hash_id == c] for c in ids]
diffs = []
for _ in range(B_ITERS):
    pick = rng.integers(0, len(blocks), len(blocks))
    y = np.concatenate([blocks[i].label_declining.to_numpy() for i in pick])
    m = np.concatenate([blocks[i].model.to_numpy() for i in pick])
    r = np.concatenate([blocks[i].action_score.to_numpy() for i in pick])
    diffs.append(p50_fast(m, y) - p50_fast(r, y))
diffs = np.array(diffs)
print(f"\n2) resampling the {len(ids)} held-out clients {B_ITERS:,} times (Week 5's own month):")
print(f"   model minus rule at precision@{TOP_K}: mean {diffs.mean():+.3f}, 5th-95th "
      f"[{np.percentile(diffs,5):+.3f}, {np.percentile(diffs,95):+.3f}] | model ahead in "
      f"{(diffs>0).mean():.1%} of resamples, behind in {(diffs<0).mean():.1%}")

# --- the blind-spot claim: uncertainty across clients vs uncertainty across months ---
bs_ids = blind_w05.client_hash_id.unique()
bs_blocks = [blind_w05[blind_w05.client_hash_id == c] for c in bs_ids]
lift_within = []
for _ in range(B_ITERS):
    pick = rng.integers(0, len(bs_blocks), len(bs_blocks))
    y = np.concatenate([bs_blocks[i].label_declining.to_numpy() for i in pick])
    m = np.concatenate([bs_blocks[i].model.to_numpy() for i in pick])
    lift_within.append(p50_fast(m, y) - y.mean())
lift_within = np.array(lift_within)
print(f"\n3) the blind-spot claim, resampling the {len(bs_ids)} held-out clients within March->April:")
print(f"   lift over the blind spot's own base rate: mean {lift_within.mean():+.3f}, 5th-95th "
      f"[{np.percentile(lift_within,5):+.3f}, {np.percentile(lift_within,95):+.3f}] | above zero in "
      f"{(lift_within>0).mean():.1%} of resamples")

blind_by_month = []
for i in range(len(PAIRS) - 1):
    ktr, kte = f"{PAIRS[i][0]}->{PAIRS[i][1]}", f"{PAIRS[i+1][0]}->{PAIRS[i+1][1]}"
    A, B = deploy(pairs[ktr], pairs[kte], TRAIN_CLIENTS, TEST_CLIENTS)
    bl = B[B.action_score == 0]
    blind_by_month.append({"deployed_on": kte, "blind_pages": len(bl),
                           "blind_base_rate": bl.label_declining.mean(),
                           "model_p50_there": p50(bl, "model"),
                           "lift": p50(bl, "model") - bl.label_declining.mean()})
blind_by_month = pd.DataFrame(blind_by_month)
print(f"\n4) the same claim, one deployment month at a time (trained on the previous pair, "
      f"judged on held-out clients):")
print(blind_by_month.round(4).to_string(index=False))

lifts = blind_by_month.lift.to_numpy()
boot_months = np.array([rng.choice(lifts, len(lifts), replace=True).mean() for _ in range(10000)])
print(f"   bootstrap over the {len(lifts)} deployment months (10,000 draws): mean {boot_months.mean():+.3f}, "
      f"5th-95th [{np.percentile(boot_months,5):+.3f}, {np.percentile(boot_months,95):+.3f}]")
print(f"   zero is inside that interval: {np.percentile(boot_months,5) < 0 < np.percentile(boot_months,95)}")
print(f"   (three windows can only produce a handful of distinct resample means, so this shows the"
      f" spread, it is not a well-powered interval)")

d_rung = ladder[ladder.protocol.str.startswith("D.")].model_minus_rule.to_numpy()
boot_d = np.array([rng.choice(d_rung, len(d_rung), replace=True).mean() for _ in range(10000)])
print(f"\n5) headline claim, across the same three deployments: model minus rule "
      f"{', '.join(f'{v:+.2f}' for v in d_rung)} | mean {d_rung.mean():+.3f}, "
      f"5th-95th [{np.percentile(boot_d,5):+.3f}, {np.percentile(boot_d,95):+.3f}]")

receipts["uncertainty"] = {
    "wilson90_on_45_of_50": [round(lo, 4), round(hi, 4)],
    "model_minus_rule_client_bootstrap": {"mean": round(float(diffs.mean()), 4),
        "p5": round(float(np.percentile(diffs, 5)), 4), "p95": round(float(np.percentile(diffs, 95)), 4),
        "share_model_ahead": round(float((diffs > 0).mean()), 4)},
    "blind_spot_lift_within_month": {"mean": round(float(lift_within.mean()), 4),
        "p5": round(float(np.percentile(lift_within, 5)), 4),
        "p95": round(float(np.percentile(lift_within, 95)), 4),
        "share_above_zero": round(float((lift_within > 0).mean()), 4)},
    "blind_spot_lift_by_deployment": {r.deployed_on: round(float(r.lift), 4)
                                      for r in blind_by_month.itertuples()},
    "blind_spot_lift_across_months": {"mean": round(float(boot_months.mean()), 4),
        "p5": round(float(np.percentile(boot_months, 5)), 4),
        "p95": round(float(np.percentile(boot_months, 95)), 4)},
    "headline_model_minus_rule_across_months": {"mean": round(float(d_rung.mean()), 4),
        "p5": round(float(np.percentile(boot_d, 5)), 4), "p95": round(float(np.percentile(boot_d, 95)), 4)}}

1) a queue of 50 is a small sample. 45 correct out of 50 = 90.0%, Wilson 90% interval [80.8%, 95.0%]
   the rule and the model are 0 points apart inside a 14-point interval.



2) resampling the 9 held-out clients 2,000 times (Week 5's own month):
   model minus rule at precision@50: mean -0.016, 5th-95th [-0.080, +0.060] | model ahead in 28.2% of resamples, behind in 51.7%



3) the blind-spot claim, resampling the 8 held-out clients within March->April:
   lift over the blind spot's own base rate: mean +0.275, 5th-95th [+0.102, +0.407] | above zero in 99.7% of resamples



4) the same claim, one deployment month at a time (trained on the previous pair, judged on held-out clients):
     deployed_on  blind_pages  blind_base_rate  model_p50_there    lift
2026-02->2026-03        17709           0.1723             0.04 -0.1323
2026-03->2026-04        16556           0.4075             0.60  0.1925
2026-04->2026-05        14211           0.2466             0.64  0.3934


   bootstrap over the 3 deployment months (10,000 draws): mean +0.152, 5th-95th [-0.024, +0.326]
   zero is inside that interval: True
   (three windows can only produce a handful of distinct resample means, so this shows the spread, it is not a well-powered interval)



5) headline claim, across the same three deployments: model minus rule -0.10, +0.08, -0.02 | mean -0.013, 5th-95th [-0.073, +0.047]


**The uncertainty sat in the dimension Week 5 did not check.**

Resampling the eight held-out clients that have blind spot pages, the lift averages 27.5 points
with a 5th to 95th range of 10.2 to 40.7, and it stays above zero in 99.7% of resamples. Inside
March to April the effect is solid, so Week 5 was not wrong about what it measured. Resampled
across months, the same lift reads -13.2, +19.3 and +39.3 points, a mean of 15.1, and the interval
of -2.4 to +32.6 contains zero. The month where it inverted is the month the panel was growing, and
there the model ranked 4% correct against a 17.2% base rate.

So the word "reproduces" was doing work the evidence could not support. It proved stability across
clients and then claimed stability in general. The headline is the same story in miniature. Model
minus rule averages -1.6 points across client resamples, with the model ahead in only 28.2% of
them, and -1.3 points across deployment months, with intervals that comfortably contain zero in
both directions. Two scorers that cannot be told apart, measured twice.

### The rewrite

**The blind spot claim, as it should read:**

> **Observed:** on pages the frozen rule scores zero, the forest's top 50 contained more next-month
> decliners than that pool's own base rate in two of three monthly deployments, and fewer in the
> third. **Measured:** 19.3 and 39.3 points of precision@50 above the local base rate in the two,
> and -13.2 points in the third; a mean of 15.1 points, with a 5th to 95th bootstrap range across
> deployments of -2.4 to +32.6 points, an interval that includes zero. Within a single month the
> effect is much steadier at 27.5 points, above zero in 99.7% of client resamples, so the
> instability lies between months rather than between clients. **Directional:** the pattern points
> toward the forest adding ranking where the rule is silent, but it did not hold in a month when
> the portfolio was growing. **Decision-support:** enough to justify running a second, smaller
> queue as a monitored trial alongside the rule's queue; not enough to promise an editor a fixed
> number of extra catches per month; and no evidence at all that reviewing these pages *causes*
> them to recover, because nothing here was an experiment.

**The headline claim, as it should read:**

> **Observed:** across three deployments from one month into the next, judged on held-out clients,
> the frozen rule and the learned model produced queues of indistinguishable quality.
> **Measured:** precision@50 of 0.88, 0.90 and 0.96 for the rule against 0.78, 0.98 and 0.94 for
> the model; a mean difference of -1.3 points with a 5th to 95th range of -7.3 to +4.7; and within
> Week 5's own month, the model ahead in 28.2% of client resamples. Against a base rate that moved
> from 19.4% to 49.4% between those months, both queues stayed far above picking at random.
> **Directional:** no consistent winner, with the rule slightly ahead on average and steadier from
> month to month. **Decision-support:** ship the rule, which is readable, protects roughly six
> times more impressions per fifty picks, and costs nothing to keep; and hold the model in the lab
> against a second queue, a per-client queue, and the sealed June month.

**What the four words are doing.** *Observed* keeps the arrow out, because pages were watched
rather than intervened on. *Measured* forces the number, the interval and the population into one
sentence. *Directional* is the honest verb for a pattern that held twice in three tries.
*Decision-support* names what the output is for, which is ordering an editor's afternoon, rather
than implying it forecasts what a page will do.

In [14]:
out_path = out_dir / "w06_validation_audit_receipts.json"
receipts["verdict"] = ("week-5 result reproduces exactly; under a time-forward split on held-out clients "
                       "the model trails the frozen rule by 1.3 points on average (interval contains zero), "
                       "and the blind-spot claim holds in 2 of 3 deployment months")
receipts["reproduced_week5"] = {"rule_p50": round(W05_RULE_P50, 4), "model_p50": round(W05_MODEL_P50, 4),
                                "model_auc": round(W05_AUC, 4)}
receipts["sources"] = ["work/outputs/w03_data_contract_receipts.json",
                       "work/outputs/w04_baseline_score_receipts.json",
                       "work/outputs/w05_model_receipts.json"]
out_path.write_text(json.dumps(receipts, indent=2), encoding="utf-8")
print(f"receipts written to work/outputs/{out_path.name} ({len(receipts)} entries)\n")
print(json.dumps({k: receipts[k] for k in ["reproduced_week5", "base_rate_by_decision_month",
                                           "population_audit", "leak_probe", "verdict"]}, indent=2))

receipts written to work/outputs/w06_validation_audit_receipts.json (19 entries)

{
  "reproduced_week5": {
    "rule_p50": 0.9,
    "model_p50": 0.9,
    "model_auc": 0.6532
  },
  "base_rate_by_decision_month": {
    "2026-01->2026-02": 0.2512,
    "2026-02->2026-03": 0.229,
    "2026-03->2026-04": 0.5378,
    "2026-04->2026-05": 0.5377
  },
  "population_audit": {
    "pages_no_next_month_row": 380,
    "share_of_positives_pct": 0.83,
    "base_rate_excluding_them": 0.5357,
    "clients_in_warehouse": 104,
    "clients_eligible": 40,
    "clients_in_slice": 27,
    "decline_rate_smallest_decile": 0.547,
    "decline_rate_largest_decile": 0.4755
  },
  "leak_probe": {
    "honest_p50": 0.9,
    "honest_auc": 0.6532,
    "LEAK_next_month_impressions": {
      "p50": 1.0,
      "auc": 0.9257
    },
    "LEAK_the_label_itself": {
      "p50": 1.0,
      "auc": 1.0
    }
  },
  "verdict": "week-5 result reproduces exactly; under a time-forward split on held-out clients the model trails t

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
